# Lecture 4 — Optimization

**Course:** UMich EECS 498-007 / 598-005 (Justin Johnson), Fall 2019, Lecture 4.  
**Merged with:** Stanford CS231n 2026 [Lecture 3: *Regularization and Optimization*](https://cs231n.stanford.edu/slides/2026/lecture_3.pdf) (the optimization part, including its appendix on Nesterov, AdaGrad, and L-BFGS).

> **About the sources.** UMich-specific content is based on knowledge of Justin Johnson's lecture; the CS231n 2026 slides were read in full. **[CS231n 2026]** marks content only there (notably **learning-rate schedules**, which UMich covers in Lecture 11); **🔎** marks material beyond both.

### How to read this notebook
* Each **code cell does one thing**; explanations come before it and **"What to notice"** after important results.
* Every function says **what it computes, what goes in, and what comes out**.
* Optimizers are first run on **tiny 2-D problems you can visualize**, then on real data. **✍️ Try it** boxes suggest experiments. Run cells **in order**.

**The goal.** Lecture 3 gave us a loss $L(W)$. **Optimization** finds
$$W^* = \arg\min_W L(W)$$
Picture the loss as a **landscape** over all possible weights, and yourself as a hiker standing somewhere on it, **blindfolded**, trying to reach the bottom of the valley. You can't see the landscape; you can only feel the slope under your feet.

| § | Topic | What you'll build |
|---|---|---|
| 1 | Idea #1: random search | Random search vs. following the slope |
| 2 | Idea #2: follow the slope; derivatives and gradients | Directional derivatives; steepest descent |
| 3 | Numeric vs. analytic gradients; **gradient checking** | The slides' finite differences; error vs. step size |
| 4 | **Gradient descent** | Too small, good, and too large learning rates |
| 5 | **Stochastic gradient descent** | Minibatch noise ∝ 1/batch size |
| 6 | Problems: poor conditioning, saddle points, noise | Zig-zags; steps ∝ condition number; a saddle |
| 7 | **Momentum** and **Nesterov** | Equivalent forms; ~√κ steps instead of κ |
| 8 | **AdaGrad, RMSProp, Adam** | Adaptive steps; aligned vs. rotated problems; Adam's bias correction |
| 9 | Optimizers on real data | Well- vs. badly-conditioned features |
| 10 | **L2 vs. weight decay; AdamW** | Same for SGD, different for Adam |
| 11 | **Learning-rate schedules** [CS231n 2026] | Step, cosine, linear, inverse sqrt, warmup |
| 12 | Second-order methods: Newton, L-BFGS | Newton in one step; L-BFGS vs. gradient descent |
| 13 | 🔎 Convexity | Same answer from every starting point |
| 14 | In practice; PyTorch | (Colab) |

### Setup

In [ ]:
import numpy as np                           # arrays and math
import matplotlib.pyplot as plt              # plots
from sklearn.datasets import load_digits     # the 8x8 handwritten digits
from scipy.optimize import minimize          # L-BFGS (Sec. 12)
import math, time

np.set_printoptions(precision=4, suppress=True)

**Data and loss** (the same as Lecture 3): digits with the mean training image subtracted and a bias feature appended, and the softmax loss with its gradient.

In [ ]:
digits = load_digits()
images, labels = digits.images / 16.0, digits.target
split = np.random.default_rng(0).permutation(len(images))
tr, va, te = split[:1000], split[1000:1300], split[1300:]
mean_image = images[tr].mean(0)

def prep(idx):
    # Flatten, subtract the mean training image, append a constant-1 (bias) feature.
    X = (images[idx] - mean_image).reshape(len(idx), -1)
    return np.hstack([X, np.ones((len(idx), 1))])

X_tr, X_va, X_te = prep(tr), prep(va), prep(te)
y_tr, y_va, y_te = labels[tr], labels[va], labels[te]
D, C = X_tr.shape[1], 10

def softmax_loss(W, X, y, reg=1e-3):
    # Average softmax (cross-entropy) loss + L2 penalty, and its gradient dL/dW (Lecture 3, Sec. 7).
    N = len(y)
    S = X @ W; S = S - S.max(1, keepdims=True)
    P = np.exp(S); P /= P.sum(1, keepdims=True)
    loss = -np.log(P[np.arange(N), y]).mean() + reg * np.sum(W * W)
    P[np.arange(N), y] -= 1                              # probabilities - one-hot targets
    return loss, X.T @ P / N + 2 * reg * W

def accuracy(W, X, y):
    return np.mean(np.argmax(X @ W, 1) == y)
print("weights to optimize:", D, "x", C, "=", D * C)

---
## 1. Idea #1: random search (a very bad idea)

Try many random weight matrices and keep the one with the lowest loss:
```python
bestloss = float("inf")
for num in range(1000):
    W = np.random.randn(10, 3073) * 0.0001     # random parameters
    loss = L(X_train, Y_train, W)              # loss over the whole training set
    if loss < bestloss:
        bestloss, bestW = loss, W
```
On CIFAR-10, the best of 1,000 random tries reaches **15.5% accuracy** (*not bad!*), against a state of the art of **~99.7%** **[CS231n 2026]**. On our digits:

In [ ]:
search_rng = np.random.default_rng(0)
best_loss, best_W = np.inf, None
t0 = time.time()
for _ in range(1000):
    W = search_rng.normal(size=(D, C))                   # a random guess
    loss, _ = softmax_loss(W, X_tr, y_tr, 0.0)
    if loss < best_loss:
        best_loss, best_W = loss, W
print(f"random search, best of 1,000 tries: loss {best_loss:.3f}, test accuracy {accuracy(best_W, X_te, y_te):.1%}  ({time.time() - t0:.1f}s)")

Compare with **100 steps of following the slope** (gradient descent, §4), a tenth as many loss evaluations:

In [ ]:
W = np.zeros((D, C))
for _ in range(100):
    loss, dW = softmax_loss(W, X_tr, y_tr, 0.0)
    W -= 0.5 * dW                                         # step downhill
print(f"gradient descent, 100 steps: loss {loss:.3f}, test accuracy {accuracy(W, X_te, y_te):.1%}")

**What to notice:** random search beats chance (10%), but following the slope is far better at a fraction of the cost. In 650 dimensions, a random guess is almost never good.

---
## 2. Idea #2: follow the slope

**In 1 dimension,** the derivative is the slope:
$$\frac{df(x)}{dx} = \lim_{h\to0}\frac{f(x + h) - f(x)}{h}$$
**In many dimensions,** the **gradient** $\nabla L$ is the vector of **partial derivatives**, one per dimension: how fast the loss changes if you move along that axis alone. Two facts make it the right tool:
1. **The slope in any direction $u$** (a unit vector) is the dot product with the gradient: $\ \text{slope along }u = u^\top\nabla L$.
2. **The direction of steepest descent is $-\nabla L$.** The dot product $u^\top\nabla L$ is smallest when $u$ points opposite to $\nabla L$, where it equals $-\|\nabla L\|$ (from $u^\top g = \|u\|\|g\|\cos\theta$).

**Check both** on a 2-D function, $f(w) = w_1^2 + 3w_2^2 + w_1w_2$, whose gradient is $\nabla f = (2w_1 + w_2,\ 6w_2 + w_1)$. **Fact 1** first:

In [ ]:
f2 = lambda w: w[0] ** 2 + 3 * w[1] ** 2 + w[0] * w[1]
grad_f2 = lambda w: np.array([2 * w[0] + w[1], 6 * w[1] + w[0]])
w0 = np.array([1.0, -0.5]); g0 = grad_f2(w0)
u = np.array([0.6, 0.8])                                   # some unit-length direction
numeric_slope = (f2(w0 + 1e-6 * u) - f2(w0 - 1e-6 * u)) / 2e-6   # how fast f changes moving along u
print(f"gradient at w0: {g0}")
print(f"slope along u: measured {numeric_slope:.6f}   vs   u . gradient = {u @ g0:.6f}")

**Fact 2:** try 3,600 directions around the circle and find the steepest downhill one.

In [ ]:
angles = np.linspace(0, 2 * np.pi, 3600, endpoint=False)
directions = np.c_[np.cos(angles), np.sin(angles)]          # 3600 unit vectors
slopes = directions @ g0                                    # slope along each one (fact 1)
steepest = directions[np.argmin(slopes)]
print(f"steepest slope found: {slopes.min():.4f}   (-|gradient| = {-np.linalg.norm(g0):.4f})")
print(f"its direction: {steepest.round(3)}   (-gradient / |gradient| = {(-g0 / np.linalg.norm(g0)).round(3)})")

---
## 3. Computing the gradient: numeric vs. analytic

### 3.1 The numeric gradient
The slides' procedure: start at the current $W$, nudge **one** dimension by a tiny $h$, see how the loss changes, and divide. E.g., $W$ = [0.34, −1.11, 0.78, …] has loss 1.25347; nudging the first entry by $h = 0.0001$ gives 1.25322, so $dW_1\approx(1.25322 - 1.25347)/0.0001 = -2.5$; the second gives 1.25353, so $dW_2\approx0.6$; the third doesn't change the loss, $dW_3 = 0$. The same on our loss:

In [ ]:
W_cur = 0.5 * np.random.default_rng(1).normal(size=(D, C))
L0, dW_analytic = softmax_loss(W_cur, X_tr, y_tr, 0.0)
h = 1e-4
print(f"current loss {L0:.5f}")
for k in range(3):
    W_h = W_cur.copy(); W_h.flat[k] += h                     # nudge entry k only
    Lh = softmax_loss(W_h, X_tr, y_tr, 0.0)[0]
    print(f"  nudge dimension {k + 1}: loss {Lh:.5f} -> dW ~ ({Lh:.5f} - {L0:.5f}) / {h} = {(Lh - L0) / h:+.4f}   "
          f"(analytic: {dW_analytic.flat[k]:+.4f})")

(Dimension 3 happens to be a blank corner pixel, which is the same for every image, so its gradient is exactly 0, like the slides' third entry.) Doing this for **all** 650 dimensions takes 1,300 loss evaluations:

In [ ]:
def numeric_gradient(f, W, h=1e-5):
    # Centered-difference estimate of the gradient of the scalar function f at W, one entry at a time.
    grad = np.zeros_like(W)
    for k in range(W.size):
        W_plus, W_minus = W.copy(), W.copy()
        W_plus.flat[k] += h; W_minus.flat[k] -= h
        grad.flat[k] = (f(W_plus) - f(W_minus)) / (2 * h)
    return grad

t0 = time.time(); g_num = numeric_gradient(lambda W_: softmax_loss(W_, X_tr, y_tr, 0.0)[0], W_cur); t_num = time.time() - t0
t0 = time.time(); softmax_loss(W_cur, X_tr, y_tr, 0.0); t_an = time.time() - t0
print(f"numeric gradient: {t_num:.2f}s;  analytic gradient: {t_an * 1000:.1f} ms  ({t_num / t_an:,.0f}x slower)")
print(f"largest difference between them: {np.abs(g_num - dW_analytic).max():.1e}")

**The numeric gradient is slow** (one or two loss evaluations per parameter; networks have millions) **and approximate**. But *the loss is just a function of $W$*, so **use calculus** to derive an **analytic gradient**: exact and fast (that's what `softmax_loss` returns).

### 3.2 Gradient checking
* **Numerical gradient:** approximate, slow, easy to write.
* **Analytic gradient:** exact, fast, error-prone.

**In practice: always use the analytic gradient, but check your implementation against the numerical one.** This is a **gradient check**. (PyTorch has `torch.autograd.gradcheck`.)

🔎 **Two details that matter.**
* **Use the centered difference.** The forward difference $\frac{f(x + h) - f(x)}{h}$ has error proportional to $h$; the centered $\frac{f(x + h) - f(x - h)}{2h}$ has error proportional to $h^2$ (in the Taylor expansions of $f(x\pm h)$, the $h^2$ terms cancel when you subtract).
* **Don't make $h$ too small.** Computers store numbers with ~16 significant digits; subtracting two nearly equal losses loses precision, and that **rounding error grows like $10^{-16}/h$**. So the total error is U-shaped in $h$:

In [ ]:
k = int(np.argmax(np.abs(dW_analytic)))                    # a weight with a large, nonzero gradient
def loss_at(t):
    # Loss as a function of that single weight, moved by t.
    W_ = W_cur.copy(); W_.flat[k] += t
    return softmax_loss(W_, X_tr, y_tr, 0.0)[0]
true_g = dW_analytic.flat[k]
hs = np.logspace(-13, -1, 25)
forward_err = [abs((loss_at(h_) - loss_at(0)) / h_ - true_g) for h_ in hs]
centered_err = [abs((loss_at(h_) - loss_at(-h_)) / (2 * h_) - true_g) for h_ in hs]

plt.figure(figsize=(6, 3.5))
plt.loglog(hs, forward_err, "o-", ms=3, label="forward difference"); plt.loglog(hs, centered_err, "s-", ms=3, label="centered difference")
plt.xlabel("h"); plt.ylabel("|numeric - analytic|"); plt.legend(); plt.title("gradient approximation error vs. step size"); plt.show()
print(f"best h: forward {hs[int(np.argmin(forward_err))]:.0e} (error {min(forward_err):.1e});  "
      f"centered {hs[int(np.argmin(centered_err))]:.0e} (error {min(centered_err):.1e})")

**What to notice:** on the right side, error falls as $h$ shrinks (the approximation gets better), faster for the centered difference; on the left, rounding takes over and error climbs again. The centered difference is far more accurate at its best $h$.

🔎 **Practical rules** (CS231n course notes): use **float64** and the **centered** difference; compare with the **relative error** $\frac{|g_a - g_n|}{\max(|g_a|, |g_n|)}$; below $10^{-7}$ is good, around $10^{-4}$ is suspicious (though it can be fine near kinks like ReLU), above $10^{-2}$ is probably a bug. Check a few random entries, and turn off randomness (like dropout) while checking.

---
## 4. Gradient descent

**Repeatedly step in the direction of the negative gradient**:
```python
w = initialize_weights()
for t in range(num_steps):
    dw = compute_gradient(loss_fn, data, w)
    w -= learning_rate * dw
```
**Hyperparameters:** how to initialize, how many steps, and the **learning rate** (step size), which matters most. Let's compare four:

In [ ]:
def gradient_descent(lr, steps=150, X=X_tr, y=y_tr):
    # Full-batch gradient descent from W = 0. Returns (final W, list of losses). Stops early if the loss blows up.
    W_ = np.zeros((X.shape[1], C)); history = []
    for _ in range(steps):
        loss, dW = softmax_loss(W_, X, y)
        history.append(loss)
        if not np.isfinite(loss) or loss > 1e3:
            break
        W_ -= lr * dW
    return W_, history

plt.figure(figsize=(6.5, 3.8))
for lr in [0.01, 0.5, 5.0, 40.0]:
    W_, history = gradient_descent(lr)
    plt.plot(history, label=f"lr = {lr}")
    print(f"lr = {lr:5}: loss after {len(history)} steps = {history[-1]:.3f}, validation accuracy {accuracy(W_, X_va, y_va):.1%}")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("training loss"); plt.legend(); plt.title("gradient descent with different learning rates"); plt.show()

**What to notice:** too small (0.01) crawls; a good one converges fast; too large (40) **overshoots** the valley and the loss bounces instead of falling. (§6.1 derives exactly how large is too large.)

---
## 5. Stochastic gradient descent (SGD)

The full-data gradient is an average over all $N$ examples:
$$\nabla_WL(W) = \frac1N\sum_{i=1}^N\nabla_WL_i(x_i, y_i, W) + \lambda\nabla_WR(W)$$
**That sum is expensive when $N$ is large!** Approximate it with a **minibatch** of examples (**32, 64, 128** are common):
```python
for t in range(num_steps):
    minibatch = sample_data(data, batch_size)
    dw = compute_gradient(loss_fn, minibatch, w)
    w -= learning_rate * dw
```
New hyperparameters: **batch size** and **how to sample** (usually: shuffle once per pass through the data, called an **epoch**).

The minibatch gradient is a **noisy but unbiased** estimate: on average it equals the full gradient, and its noise shrinks as the batch grows. 🔎 Theory says the noise **variance** falls as $1/B$. Let's measure it. **Step 1:** the full gradient at some point partway through training.

In [ ]:
W_mid = gradient_descent(0.5, steps=20)[0]
_, g_full = softmax_loss(W_mid, X_tr, y_tr)
print("full-batch gradient norm:", round(float(np.linalg.norm(g_full)), 4))

**Step 2:** for each batch size, sample 400 minibatches and measure how far their gradients are from the full gradient, on average ($E\|g_B - g\|^2$). If the variance is $\propto1/B$, then $B\times E\|g_B - g\|^2$ should be roughly constant.

In [ ]:
noise_rng = np.random.default_rng(3)
print(f"{'batch size':>10s} {'E|g_B - g|^2':>14s} {'x batch size':>14s}")
for B in [1, 4, 16, 64, 256]:
    errors = []
    for _ in range(400):
        idx = noise_rng.integers(0, len(X_tr), B)
        g_B = softmax_loss(W_mid, X_tr[idx], y_tr[idx])[1]
        errors.append(np.sum((g_B - g_full) ** 2))
    print(f"{B:10d} {np.mean(errors):14.5f} {np.mean(errors) * B:14.5f}")

**What to notice:** the last column is constant: variance ∝ 1/B, so the noise's **size** (standard deviation) falls only as $1/\sqrt B$. **Quadrupling the batch halves the noise** at 4× the computation per step. That's why moderate batch sizes are efficient. *Interactive demo:* http://vision.stanford.edu/teaching/cs231n-demos/linear-classify/

---
## 6. Problems with SGD

### 6.1 Problem #1: poor conditioning
**What if the loss changes quickly in one direction and slowly in another?** Gradient descent makes **very slow progress along the shallow direction and jitters along the steep one.** We say the loss has a high **condition number**: the ratio of the largest to smallest singular value of its **Hessian** (the matrix of second derivatives, i.e., the curvatures).

🔎 **A quadratic makes this exact.** Take $L(w) = \tfrac12w^\top Aw$, where $A$ has eigenvalues (curvatures) $\lambda_{\min}\leq\dots\leq\lambda_{\max}$. Its gradient is $Aw$, so a gradient step is $w\leftarrow(I - \alpha A)w$, and along each eigenvector the error is multiplied by $(1 - \alpha\lambda_i)$ every step:
* To not blow up, every factor must have size < 1, which needs $\alpha < 2/\lambda_{\max}$. **The steepest direction limits the learning rate.**
* With such an $\alpha$, the flattest direction shrinks by only $1 - \alpha\lambda_{\min}\approx1 - 2/\kappa$ per step, where $\kappa = \lambda_{\max}/\lambda_{\min}$. **The number of steps grows in proportion to the condition number.**

**Step 1: a test problem factory.** `quad(kappa, angle)` builds a 2-D quadratic with curvatures 1 and κ, optionally rotated, and returns the loss, its gradient, and the matrix $A$.

In [ ]:
def quad(kappa, angle=0.0):
    # 2-D quadratic L(w) = 0.5 w^T A w with eigenvalues 1 and kappa, rotated by `angle`.
    # Returns (loss function, gradient function, A).
    A = np.diag([1.0, kappa])
    c, s_ = np.cos(angle), np.sin(angle); R = np.array([[c, -s_], [s_, c]])
    A = R @ A @ R.T
    return (lambda w: 0.5 * w @ A @ w), (lambda w: A @ w), A

**Step 2: a small framework for optimizers.** Each optimizer is a function `step(w, g, state, t, grad_fn)` that returns the new `w`; `state` is a dictionary where it can keep things like velocity between steps. `run` applies it repeatedly and records the path. Plain SGD is the simplest one:

In [ ]:
def run(optimizer_step, grad_fn, w0, steps):
    # Apply an optimizer for `steps` iterations from w0. Returns the path, an array of shape (steps + 1, dim).
    w, state, path = w0.copy(), {}, [w0.copy()]
    for t in range(1, steps + 1):
        w = optimizer_step(w, grad_fn(w), state, t, grad_fn)
        path.append(w.copy())
    return np.array(path)

def sgd(lr):
    # Plain gradient descent: w <- w - lr * g.
    return lambda w, g, state, t, grad_fn: w - lr * g

**Step 3: watch gradient descent zig-zag** on a quadratic with κ = 20, with the learning rate just under the limit $2/\lambda_{\max} = 0.1$:

In [ ]:
f_q, g_q, A_q = quad(20.0)
w_start = np.array([-9.0, 1.6])
xs, ys = np.meshgrid(np.linspace(-10, 3, 200), np.linspace(-2.2, 2.2, 200))
contours = 0.5 * (A_q[0, 0] * xs ** 2 + 2 * A_q[0, 1] * xs * ys + A_q[1, 1] * ys ** 2)

plt.figure(figsize=(7, 3.5)); plt.contour(xs, ys, contours, levels=20, colors="lightgray")
path = run(sgd(0.095), g_q, w_start, 40)
plt.plot(*path.T, "o-", ms=3, label="gradient descent, lr 0.095")
plt.legend(fontsize=8); plt.title("condition number 20: jitter across, slow progress along"); plt.show()

**Step 4: count steps vs. condition number**, each with the best constant learning rate for gradient descent, $2/(\lambda_{\max} + \lambda_{\min})$:

In [ ]:
print(f"{'kappa':>7s} {'GD steps to reach loss < 1e-8':>32s}")
for kappa in [10, 100, 1000]:
    f_, g_, A_ = quad(kappa)
    lr, w, n = 2 / (kappa + 1), np.array([1.0, 1.0]), 0
    while f_(w) > 1e-8:
        w = w - lr * g_(w); n += 1
    print(f"{kappa:7d} {n:32d}")

**What to notice:** 10× the condition number → about 10× the steps, exactly as the analysis predicts. Real networks' losses are badly conditioned, which motivates everything in §7–8.

### 6.2 Problem #2: local minima and saddle points
**What if the loss has a local minimum or a saddle point?** The gradient is **zero** there, and gradient descent **gets stuck**. **Saddle points are much more common in high dimensions** (Dauphin et al., NeurIPS 2014): at a point where the gradient is zero in a million-dimensional space, it's far more likely that *some* directions curve up and some curve down (a saddle) than that *all* curve up (a minimum). Near a saddle the gradient is tiny, so progress is slow even if you're not exactly stuck.

**Test function:** $f(x, y) = x^2 - y^2 + \tfrac14y^4$: a saddle at $(0, 0)$ (curving up along $x$, down along $y$), with minima at $(0, \pm\sqrt2)$ where $f = -1$. We start at $x = 1$ and a tiny distance $y_0$ off the saddle's "ridge":

In [ ]:
saddle_f = lambda w: w[0] ** 2 - w[1] ** 2 + 0.25 * w[1] ** 4
saddle_g = lambda w: np.array([2 * w[0], -2 * w[1] + w[1] ** 3])
for y0 in [1e-1, 1e-4, 1e-8, 0.0]:
    w, n = np.array([1.0, y0]), 0
    while saddle_f(w) > -0.99 and n < 5000:            # until we're (almost) at a minimum
        w = w - 0.05 * saddle_g(w); n += 1
    print(f"start at (1, {y0:g}): " + (f"reached a minimum after {n} steps" if n < 5000 else "STUCK at the saddle point after 5,000 steps"))

**What to notice:** every factor of 10 closer to the ridge costs roughly two dozen extra steps (the escape speeds up exponentially, so the delay grows with the *logarithm* of the distance). Starting exactly on it, plain gradient descent never leaves. Noise (SGD) or momentum helps push off.

### 6.3 Problem #3: noise
**Minibatch gradients are noisy!** (§5.) The noise makes the path wander and keeps it bouncing around near the minimum, unless the learning rate decreases over time (§11).

---
## 7. SGD + Momentum

**Keep moving in the general direction of previous steps.** Build up a **velocity** $v$ as a running average of gradients; $\rho$ acts like **friction** (typically **0.9 or 0.99**; Sutskever et al., ICML 2013):
$$v_{t+1} = \rho\,v_t + \nabla f(x_t),\qquad x_{t+1} = x_t - \alpha\,v_{t+1}$$
**An alternative, equivalent form:** $v_{t+1} = \rho\,v_t - \alpha\nabla f(x_t)$, $\ x_{t+1} = x_t + v_{t+1}$. *You'll see it written both ways; they give the same sequence of $x$* (the second $v$ is just $-\alpha$ times the first).

Why it helps:
* **Poor conditioning:** zig-zags across the valley cancel out in the running average; consistent motion along the valley accumulates.
* **Saddles and local minima:** velocity carries it through flat spots.
* **Noise:** averaging smooths the gradients.

**Both forms, as optimizers:**

In [ ]:
def momentum(lr, rho):
    # SGD + momentum, form 1: v <- rho v + g;  w <- w - lr v
    def step(w, g, state, t, grad_fn):
        state["v"] = rho * state.get("v", 0) + g
        return w - lr * state["v"]
    return step

def momentum_alt(lr, rho):
    # SGD + momentum, form 2: v <- rho v - lr g;  w <- w + v
    def step(w, g, state, t, grad_fn):
        state["v"] = rho * state.get("v", 0) - lr * g
        return w + state["v"]
    return step

p1 = run(momentum(0.02, 0.9), g_q, w_start, 50)
p2 = run(momentum_alt(0.02, 0.9), g_q, w_start, 50)
print("the two forms give the same sequence of points:", np.allclose(p1, p2))

**Compare with plain gradient descent** on the κ = 20 quadratic, 40 steps each:

In [ ]:
plt.figure(figsize=(7, 3.5)); plt.contour(xs, ys, contours, levels=20, colors="lightgray")
p_sgd = run(sgd(0.095), g_q, w_start, 40)
p_mom = run(momentum(0.02, 0.8), g_q, w_start, 40)
plt.plot(*p_sgd.T, "o-", ms=3, label=f"SGD: loss after 40 steps {f_q(p_sgd[-1]):.2e}")
plt.plot(*p_mom.T, "s-", ms=3, label=f"SGD + momentum (rho 0.8): {f_q(p_mom[-1]):.2e}")
plt.legend(fontsize=8); plt.show()

🔎 **How much faster, in theory?** For a quadratic with condition number κ, the best constant step for gradient descent needs about **κ** steps per factor-of-$e$ error reduction; **momentum (the "heavy ball" method) with the best settings needs about √κ** (Polyak 1964). The best settings are
$$\alpha = \frac{4}{(\sqrt{\lambda_{\max}} + \sqrt{\lambda_{\min}})^2},\qquad\rho = \left(\frac{\sqrt\kappa - 1}{\sqrt\kappa + 1}\right)^2$$
Let's count:

In [ ]:
print(f"{'kappa':>7s} {'GD steps':>9s} {'momentum steps':>15s} {'sqrt(kappa)':>12s}")
for kappa in [10, 100, 1000, 10000]:
    f_, g_, A_ = quad(kappa)
    w, n_gd = np.array([1.0, 1.0]), 0
    while f_(w) > 1e-8:
        w = w - 2 / (kappa + 1) * g_(w); n_gd += 1
    lr = 4 / (1 + math.sqrt(kappa)) ** 2
    rho = ((math.sqrt(kappa) - 1) / (math.sqrt(kappa) + 1)) ** 2
    w, v, n_m = np.array([1.0, 1.0]), np.zeros(2), 0
    while f_(w) > 1e-8:
        v = rho * v - lr * g_(w); w = w + v; n_m += 1
    print(f"{kappa:7d} {n_gd:9d} {n_m:15d} {math.sqrt(kappa):12.0f}")

**What to notice:** for κ = 10,000, momentum needs about one sixtieth of the steps, and its step count grows roughly like √κ. (Those are ideal settings for a known quadratic; in practice ρ = 0.9 is a robust default, and the speedup is smaller but real.)

### 7.1 Nesterov momentum
**"Look ahead"** to where the velocity would carry us, compute the gradient **there**, and use it to correct the velocity (Nesterov 1983; Sutskever et al. 2013):
$$v_{t+1} = \rho\,v_t - \alpha\nabla f(x_t + \rho\,v_t),\qquad x_{t+1} = x_t + v_{t+1}$$
*Annoying:* we'd rather have an update in terms of $x_t$ and $\nabla f(x_t)$. **Change variables** to the look-ahead point $\tilde x_t = x_t + \rho v_t$ and rearrange:
$$v_{t+1} = \rho\,v_t - \alpha\nabla f(\tilde x_t),\qquad\tilde x_{t+1} = \tilde x_t - \rho\,v_t + (1 + \rho)\,v_{t+1}$$
**Check that both forms describe the same algorithm**: run each, and confirm that form 2's $\tilde x_t$ equals form 1's $x_t + \rho v_t$ at every step.

In [ ]:
lr, rho = 0.02, 0.9
x, v = w_start.copy(), np.zeros(2); form1_lookahead = []
for t in range(60):                                         # form 1: gradient at the look-ahead point
    form1_lookahead.append(x + rho * v)
    v = rho * v - lr * g_q(x + rho * v)
    x = x + v
xt, v2 = w_start.copy(), np.zeros(2); form2_points = []
for t in range(60):                                         # form 2: change of variables
    form2_points.append(xt.copy())
    v_new = rho * v2 - lr * g_q(xt)
    xt = xt - rho * v2 + (1 + rho) * v_new
    v2 = v_new
print("form 2's x~_t equals form 1's x_t + rho v_t at every step:", np.allclose(form1_lookahead, form2_points))
p_m = run(momentum_alt(lr, rho), g_q, w_start, 60)
print(f"loss after 60 steps (lr {lr}, rho {rho}): momentum {f_q(p_m[-1]):.2e}, Nesterov {f_q(x):.2e}")

**What to notice:** with the same settings, Nesterov's look-ahead corrects the velocity before it overshoots, so it oscillates less and gets lower. (For convex problems it also has a better worst-case rate.) As an optimizer function, for later comparisons:

In [ ]:
def nesterov(lr, rho):
    # Nesterov momentum (form 1): gradient evaluated at the look-ahead point w + rho v.
    def step(w, g, state, t, grad_fn):
        v = state.get("v", np.zeros_like(w))
        state["v"] = rho * v - lr * grad_fn(w + rho * v)
        return w + state["v"]
    return step

---
## 8. Adaptive learning rates: AdaGrad, RMSProp, Adam

### 8.1 AdaGrad (Duchi et al., 2011)
Scale each parameter's step by the **history of its own squared gradients** ("per-parameter" or "adaptive" learning rates):
```python
grad_squared = 0
while True:
    dx = compute_gradient(x)
    grad_squared += dx * dx
    x -= learning_rate * dx / (np.sqrt(grad_squared) + 1e-7)
```
* **Q: what happens?** Steps along **steep** directions (big gradients) are **damped**; steps along **flat** ones are **accelerated**.
* **Q2: over a long time?** `grad_squared` only grows, so the steps **decay to zero**.

### 8.2 RMSProp: "leaky AdaGrad" (Tieleman & Hinton, 2012)
Use a **decaying average** of squared gradients, so old gradients are forgotten and steps don't vanish:
```python
grad_squared = decay_rate * grad_squared + (1 - decay_rate) * dx * dx
x -= learning_rate * dx / (np.sqrt(grad_squared) + 1e-7)
```

### 8.3 Adam (Kingma & Ba, 2015): momentum + RMSProp + bias correction
```python
moment1 = moment2 = 0
for t in range(1, num_iterations + 1):
    dx = compute_gradient(x)
    moment1 = beta1 * moment1 + (1 - beta1) * dx           # momentum (running mean of gradients)
    moment2 = beta2 * moment2 + (1 - beta2) * dx * dx      # RMSProp (running mean of squared gradients)
    moment1_unbias = moment1 / (1 - beta1 ** t)            # bias correction
    moment2_unbias = moment2 / (1 - beta2 ** t)
    x -= learning_rate * moment1_unbias / (np.sqrt(moment2_unbias) + 1e-7)
```
**Adam with β1 = 0.9, β2 = 0.999, and learning rate 1e-3 or 5e-4 is a great starting point for many models!**

The three as optimizer functions:

In [ ]:
def adagrad(lr):
    # AdaGrad: divide by the root of the SUM of all past squared gradients.
    def step(w, g, state, t, grad_fn):
        state["s"] = state.get("s", 0) + g * g
        return w - lr * g / (np.sqrt(state["s"]) + 1e-7)
    return step

def rmsprop(lr, decay=0.9):
    # RMSProp: divide by the root of a DECAYING AVERAGE of squared gradients.
    def step(w, g, state, t, grad_fn):
        state["s"] = decay * state.get("s", 0) + (1 - decay) * g * g
        return w - lr * g / (np.sqrt(state["s"]) + 1e-7)
    return step

def adam(lr, b1=0.9, b2=0.999):
    # Adam: momentum on the gradient, RMSProp scaling, and bias correction for both.
    def step(w, g, state, t, grad_fn):
        state["m"] = b1 * state.get("m", 0) + (1 - b1) * g
        state["v"] = b2 * state.get("v", 0) + (1 - b2) * g * g
        m_hat, v_hat = state["m"] / (1 - b1 ** t), state["v"] / (1 - b2 ** t)
        return w - lr * m_hat / (np.sqrt(v_hat) + 1e-7)
    return step

### 8.4 Why Adam needs bias correction
**Q: What happens at the first timestep** without it? Both moments start at **zero**, so after one step they're $(1 - \beta_1)g$ and $(1 - \beta_2)g^2$: shrunk toward zero, the second far more ($1 - \beta_2 = 0.001$ vs. $0.1$). The step is then
$$\frac{(1 - \beta_1)g}{\sqrt{(1 - \beta_2)g^2}} = \frac{0.1}{\sqrt{0.001}}\ \text{sign}(g)\approx3.16\ \text{sign}(g)$$
about **3× too large**, and if a gradient is near zero at the start, the tiny second moment can make the step huge. Dividing by $1 - \beta^t$ undoes the bias from starting at zero. Measured with a constant gradient of 0.5:

In [ ]:
def adam_step_sizes(gradients, lr=1e-3, b1=0.9, b2=0.999, correct=True):
    # The size of each Adam step for a given sequence of gradients, with or without bias correction.
    m = v = 0.0; sizes = []
    for t, g in enumerate(gradients, 1):
        m = b1 * m + (1 - b1) * g; v = b2 * v + (1 - b2) * g * g
        m_hat, v_hat = (m / (1 - b1 ** t), v / (1 - b2 ** t)) if correct else (m, v)
        sizes.append(lr * m_hat / (np.sqrt(v_hat) + 1e-7))
    return np.array(sizes)

constant_gradient = np.full(10, 0.5)
print("step sizes (in units of lr), constant gradient 0.5:")
print("  with bias correction   :", (adam_step_sizes(constant_gradient) / 1e-3).round(3))
print("  without bias correction:", (adam_step_sizes(constant_gradient, correct=False) / 1e-3).round(3))
print(f"  first-step ratio 0.1 / sqrt(0.001) = {0.1 / math.sqrt(0.001):.3f}")

**What to notice:** with correction, Adam's step for a steady gradient is about one learning rate from the very first step. Also note a key property: **Adam's step size is roughly the learning rate whatever the gradient's scale** (it divides by the gradient's own typical size), so the learning rate directly controls how far each parameter moves per step.

### 8.5 Adaptive methods on aligned vs. rotated problems
AdaGrad, RMSProp, and Adam rescale **each coordinate separately**. That's ideal when the steep and flat directions line up with the **coordinate axes**, and less helpful when they're **rotated** (when the hard directions are combinations of parameters). 🔎 To isolate the effect of rotation, we compare a κ = 20 quadratic with **the exact same problem rotated by 45°**, starting point included. For each optimizer, we pick its best learning rate from a grid and count the steps to reach 1% of the starting loss.

**Step 1: the "best learning rate" search.**

In [ ]:
makers = {"SGD": sgd, "Momentum": lambda lr: momentum(lr, 0.8), "Nesterov": lambda lr: nesterov(lr, 0.8),
          "AdaGrad": adagrad, "RMSProp": rmsprop, "Adam": adam}
lr_grid = [0.003, 0.01, 0.03, 0.09, 0.3, 1.0, 3.0]

def fastest(make_optimizer, f_, g_, w0, steps=60):
    # Over the learning-rate grid, find the run that reaches 1% of the starting loss in the fewest steps.
    # Returns (steps needed, learning rate, loss after `steps` steps, path) or None if no run gets there.
    best = None
    for lr in lr_grid:
        with np.errstate(all="ignore"):
            path = run(make_optimizer(lr), g_, w0, steps)
        losses = [f_(w) for w in path]
        hit = next((i for i, v in enumerate(losses) if v < 0.01 * losses[0]), None)
        if hit is not None and (best is None or hit < best[0]):
            best = (hit, lr, losses[-1], path)
    return best

**Step 2: run every optimizer on both problems.**

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for a, angle, title in [(ax[0], 0.0, "axis-aligned (kappa 20)"), (ax[1], np.pi / 4, "rotated 45 degrees (kappa 20)")]:
    f_, g_, A_ = quad(20.0, angle)
    Rot = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    w0_ = Rot @ np.array([-9.0, 1.6])                       # rotate the start too: the SAME problem, just rotated
    gx2, gy2 = np.meshgrid(np.linspace(-10, 4, 200), np.linspace(-8, 4, 200))
    a.contour(gx2, gy2, 0.5 * (A_[0, 0] * gx2 ** 2 + 2 * A_[0, 1] * gx2 * gy2 + A_[1, 1] * gy2 ** 2), levels=20, colors="lightgray")
    print(f"{title}: steps to reach 1% of the starting loss (best lr), and loss after 60 steps")
    for name, mk in makers.items():
        result = fastest(mk, f_, g_, w0_)
        if result is None:
            print(f"   {name:9s}: not within 60 steps"); continue
        hit, lr, final, path = result
        a.plot(*path[:hit + 1].T, "-", lw=1.2, label=f"{name} ({hit})")
        print(f"   {name:9s}: {hit:3d} steps (lr {lr:g});  loss after 60 steps {final:.1e}")
    a.set_title(title); a.legend(fontsize=7); a.set_xlim(-10, 4); a.set_ylim(-8, 4); a.set_aspect("equal")
plt.show()

**What to notice:**
* **SGD, momentum, and Nesterov take exactly the same number of steps on both problems.** Their updates are built from the gradient alone, which rotates along with the problem, so they only "see" the curvature, not the axes (they're **rotation-equivariant**).
* **The adaptive methods change.** On the aligned problem, AdaGrad and RMSProp are the fastest of all (per-coordinate scaling undoes the conditioning). On the rotated one they slow down sharply (about 5× more steps), because the steep direction is now a mix of both coordinates. **Adam** is barely affected here: it also has momentum, and its long memory (β₂ = 0.999) makes its scaling nearly uniform on this problem.
* **The final losses:** with a constant learning rate, Adam and RMSProp keep taking steps of about one learning rate and **hover around the minimum** instead of settling. That's what learning-rate decay (§11) fixes.

Real networks are in between: different parameters do have very different gradient scales, often along the axes (e.g., biases vs. weights, early vs. late layers), which is a big part of why Adam works so well out of the box.

| Optimizer | Momentum (1st moment)? | Adaptive (2nd moment)? | Leaky 2nd moment? | Bias correction? |
|---|---|---|---|---|
| SGD | ✗ | ✗ | ✗ | ✗ |
| SGD + Momentum | ✓ | ✗ | ✗ | ✗ |
| Nesterov | ✓ | ✗ | ✗ | ✗ |
| AdaGrad | ✗ | ✓ | ✗ | ✗ |
| RMSProp | ✗ | ✓ | ✓ | ✗ |
| Adam | ✓ | ✓ | ✓ | ✓ |

---
## 9. Comparing optimizers on real data

The softmax classifier on digits, with minibatches of 64 for 300 steps, and each optimizer's best learning rate from a grid. Two versions of the same problem:
* **Well-conditioned:** our usual inputs (mean-subtracted pixels in [−1, 1]).
* **Badly conditioned:** raw pixel values 0–16, not centered, plus a constant 1 (features on very different scales make the loss badly conditioned).

**Step 1: a minibatch training loop** that accepts any of our optimizers.

In [ ]:
def train_minibatch(make_opt, X, y, steps=300, batch=64, seed=0):
    # Train the softmax classifier from W = 0 with the given optimizer. Returns (full training loss, W).
    r = np.random.default_rng(seed)
    W_ = np.zeros((X.shape[1], C)); state = {}
    for t in range(1, steps + 1):
        idx = r.integers(0, len(X), batch)
        g = softmax_loss(W_, X[idx], y[idx])[1]
        W_ = make_opt(W_, g, state, t, lambda w: softmax_loss(w, X[idx], y[idx])[1])
    return softmax_loss(W_, X, y)[0], W_

X_bad_tr = np.hstack([images[tr].reshape(len(tr), -1) * 16, np.ones((len(tr), 1))])   # raw 0..16, uncentered
X_bad_va = np.hstack([images[va].reshape(len(va), -1) * 16, np.ones((len(va), 1))])
print("feature scales: well-conditioned std", X_tr[:, :64].std().round(2), "| raw pixels std", X_bad_tr[:, :64].std().round(2))

**Step 2: run all six on both versions.**

In [ ]:
grids = {"SGD": sgd, "Momentum (0.9)": lambda lr: momentum(lr, 0.9), "Nesterov (0.9)": lambda lr: nesterov(lr, 0.9),
         "AdaGrad": adagrad, "RMSProp": lambda lr: rmsprop(lr, 0.99), "Adam": adam}
lrs = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
print(f"{'optimizer':16s} {'well-conditioned: loss (val acc) [lr]':>38s} {'badly conditioned: loss (val acc) [lr]':>41s}")
for name, mk in grids.items():
    cells = []
    for Xa, Xv in [(X_tr, X_va), (X_bad_tr, X_bad_va)]:
        results = []
        for lr in lrs:
            with np.errstate(all="ignore"):
                loss, W_ = train_minibatch(mk(lr), Xa, y_tr)
            if np.isfinite(loss):
                results.append((loss, accuracy(W_, Xv, y_va), lr))
        l_, a_, lr_ = min(results)
        cells.append(f"{l_:.3f} ({a_:.1%}) [{lr_:g}]")
    print(f"{name:16s} {cells[0]:>38s} {cells[1]:>41s}")

**What to notice:** compare optimizers **within a column** (the two columns are different problems, and unscaled pixels also change how strong the fixed regularization is, so losses across columns aren't comparable). Well-conditioned: every method reaches nearly the same loss once its learning rate is tuned. Badly conditioned: plain SGD ends with a loss two to three times higher than the others; momentum, Nesterov, and the adaptive methods all cope much better. Two lessons: **preprocess your inputs** (centering and scaling help every optimizer), and **momentum and adaptive methods are more forgiving** when you can't.

---
## 10. L2 regularization vs. weight decay: AdamW

**Q: How does regularization interact with the optimizer (e.g., L2)? A: It depends!** **[CS231n 2026]**
* **L2 regularization** adds $\lambda\|w\|^2$ to the loss, so its gradient $2\lambda w$ is added to the data gradient **before** the optimizer sees it.
* **Weight decay** shrinks the weights directly in the update: $w\leftarrow w - \alpha(\text{optimizer step}) - \alpha\lambda'w$.

**For SGD they're identical** (with $\lambda' = 2\lambda$). **For Adam they're not:** the L2 gradient goes into Adam's moment estimates and gets **divided by $\sqrt v$** like everything else, so parameters with large gradients are barely regularized. **AdamW** (Loshchilov & Hutter, 2019) applies the decay **after** the moment calculations, outside the normalization, so every parameter decays at the same rate. It tends to generalize better (fast.ai's ImageNet comparison in the slides) and is the standard for training Transformers.

**Test problem:** two parameters, both with optimum 1, but very different gradient scales ($50(w_1 - 1)$ vs. $0.05(w_2 - 1)$). **Step 1: SGD, L2 vs. weight decay.**

In [ ]:
g_data = lambda w: np.array([50.0, 0.05]) * (w - 1.0)       # data gradient: parameter 1 steep, parameter 2 very flat
w_l2, w_wd = np.zeros(2), np.zeros(2)
for _ in range(200):
    w_l2 = w_l2 - 0.01 * (g_data(w_l2) + 2 * 0.1 * w_l2)      # SGD + L2 penalty (lambda = 0.1): gradient of 0.1 ||w||^2
    w_wd = w_wd - 0.01 * g_data(w_wd) - 0.01 * 0.2 * w_wd      # SGD + weight decay (lambda' = 0.2)
print("SGD: L2 regularization and weight decay give identical weights:", np.allclose(w_l2, w_wd), w_l2.round(4))

**Step 2: Adam + L2 vs. AdamW.**

In [ ]:
def adam_run(decoupled, wd=0.2, lr=0.01, steps=5000):
    # Adam with an L2 penalty (decoupled=False) or with decoupled weight decay (AdamW, decoupled=True).
    w, m, v = np.zeros(2), np.zeros(2), np.zeros(2)
    for t in range(1, steps + 1):
        g = g_data(w) + (0 if decoupled else wd * w)            # Adam + L2: the penalty goes INTO the moments
        m = 0.9 * m + 0.1 * g; v = 0.999 * v + 0.001 * g * g
        w = w - lr * (m / (1 - 0.9 ** t)) / (np.sqrt(v / (1 - 0.999 ** t)) + 1e-8)
        if decoupled:
            w = w - lr * wd * w                                 # AdamW: decay applied directly, after the moments
    return w

exact_l2_optimum = np.array([50, 0.05]) / (np.array([50, 0.05]) + 0.2)
print(f"Adam + L2: w = {adam_run(False).round(3)}   (the exact minimizer of the L2-regularized loss: {exact_l2_optimum.round(3)})")
print(f"AdamW    : w = {adam_run(True).round(3)}")

**What to notice:**
* **SGD:** L2 and weight decay are the same algorithm.
* **Adam + L2** ends at the minimizer of the L2-regularized loss, where the penalty's pull is weighed against each parameter's **data gradient**: the steep parameter is barely shrunk, the flat one is pulled most of the way to zero.
* **AdamW** shrinks both parameters by the same relative amount, regardless of their gradient scale (here both end at 0.996): a different solution. For deep networks, AdamW's behavior has proven easier to tune (learning rate and decay strength stop interacting), which is why it became the default.

---
## 11. Learning-rate schedules [CS231n 2026]

SGD, momentum, RMSProp, Adam, and AdamW all have a **learning rate** hyperparameter. **Q: which learning rate is best?** Plotting loss vs. epoch for "very high", "high", "good", and "low" rates suggests "good", but **in reality, all of these could be good learning rates**, at different times: **start high, then decay**. Common schedules ($\alpha_0$ = initial rate, $\alpha_t$ = rate at epoch $t$, $T$ = total epochs):
* **Step:** reduce at a few fixed points; e.g., for ResNets, multiply by 0.1 after epochs 30, 60, and 90.
* **Cosine:** $\alpha_t = \tfrac12\alpha_0\left(1 + \cos(t\pi/T)\right)$ (Loshchilov & Hutter 2017; widely used).
* **Linear:** $\alpha_t = \alpha_0(1 - t/T)$ (e.g., BERT).
* **Inverse square root:** $\alpha_t = \alpha_0/\sqrt t$ (the original Transformer).
* **Linear warmup:** a high initial rate can make the loss explode, so **increase it linearly from 0 over the first ~5,000 iterations**. Rule of thumb: **if you increase the batch size by N, scale the initial learning rate by N** (Goyal et al. 2017).

**Step 1: the schedules as functions** (each returns the multiplier of $\alpha_0$ at epoch $t$):

In [ ]:
T = 100
schedules = {
    "constant": lambda t: 1.0,
    "step (x0.1 at 30, 60, 90)": lambda t: 0.1 ** sum(t >= m for m in (30, 60, 90)),
    "cosine": lambda t: 0.5 * (1 + math.cos(t * math.pi / T)),
    "linear": lambda t: 1 - t / T,
    "inverse sqrt": lambda t: 1 / math.sqrt(max(t, 1)),
    "warmup (5) + cosine": lambda t: (t + 1) / 5 if t < 5 else 0.5 * (1 + math.cos((t - 5) * math.pi / (T - 5))),
}
plt.figure(figsize=(7, 3.5))
for name, f_ in schedules.items():
    plt.plot([f_(t) for t in range(T)], label=name)
plt.xlabel("epoch"); plt.ylabel("learning rate / initial"); plt.legend(fontsize=7); plt.show()

**Step 2: does it matter?** Train with a large learning rate (5) and small batches (8), where minibatch noise is strong, for 100 epochs with each schedule:

In [ ]:
def train_with_schedule(schedule, lr0=5.0, epochs=T, batch=8, seed=0):
    # SGD with the learning rate set by schedule(epoch) * lr0. Returns (final training loss, validation accuracy).
    r = np.random.default_rng(seed); W_ = np.zeros((D, C))
    for epoch in range(epochs):
        lr = lr0 * schedule(epoch)
        for idx in np.array_split(r.permutation(len(X_tr)), len(X_tr) // batch):     # one shuffled pass
            W_ -= lr * softmax_loss(W_, X_tr[idx], y_tr[idx])[1]
    return softmax_loss(W_, X_tr, y_tr)[0], accuracy(W_, X_va, y_va)

for name in ["constant", "step (x0.1 at 30, 60, 90)", "cosine", "linear"]:
    loss, acc = train_with_schedule(schedules[name])
    print(f"{name:28s}: final training loss {loss:.4f}, validation accuracy {acc:.1%}")

**What to notice:** with a constant large rate, the noise keeps the weights bouncing around the minimum, so the final loss stays high; every decaying schedule lets the weights settle to essentially the same, lower loss. (With a small rate or large batches the noise is small and schedules barely differ: ✍️ try `lr0=0.5, batch=32`.)

---
## 12. Second-order optimization

* **First-order:** use the gradient to make a **linear** approximation of the loss, and step to reduce it.
* **Second-order:** use the gradient **and the Hessian** to make a **quadratic** approximation, and step to its **minimum**.

The second-order Taylor expansion around $w_0$:
$$L(w)\approx L(w_0) + (w - w_0)^\top\nabla L(w_0) + \tfrac12(w - w_0)^\top H(w_0)\,(w - w_0)$$
Setting its gradient to zero gives the **Newton step**:
$$w^* = w_0 - H(w_0)^{-1}\nabla L(w_0)$$
No learning rate, and on a quadratic it lands exactly on the minimum **in one step**, however badly conditioned:

In [ ]:
for kappa in [20, 1e6]:
    f_, g_, A_ = quad(kappa, np.pi / 5)
    w0 = np.array([-9.0, 1.6])
    w1 = w0 - np.linalg.solve(A_, g_(w0))              # Newton step (the Hessian of 0.5 w^T A w is A)
    print(f"kappa = {kappa:g}: loss {f_(w0):.3g} -> {f_(w1):.1e} after ONE Newton step")

**On a real problem:** binary logistic regression (digits 3 vs. 8), with a little L2 so the minimum is finite. The function below returns the loss, gradient, and Hessian $H = X^\top\text{diag}(p(1-p))X/N + 2\lambda I$.

In [ ]:
mask = np.isin(y_tr, [3, 8])
Xb, yb = X_tr[mask], (y_tr[mask] == 8).astype(float); lam = 1e-2
def logistic(w):
    # Regularized logistic regression: returns (loss, gradient, Hessian).
    p_ = 1 / (1 + np.exp(-Xb @ w))
    loss = -np.mean(yb * np.log(p_ + 1e-12) + (1 - yb) * np.log(1 - p_ + 1e-12)) + lam * w @ w
    grad = Xb.T @ (p_ - yb) / len(yb) + 2 * lam * w
    hess = (Xb.T * (p_ * (1 - p_))) @ Xb / len(yb) + 2 * lam * np.eye(len(w))
    return loss, grad, hess

w = np.zeros(D)
for it in range(1, 20):
    loss, grad, hess = logistic(w)
    if np.linalg.norm(grad) < 1e-10: break
    w = w - np.linalg.solve(hess, grad)                 # Newton step
print(f"Newton: gradient norm below 1e-10 after {it - 1} iterations (loss {loss:.6f})")
w, n = np.zeros(D), 0
while np.linalg.norm(logistic(w)[1]) > 1e-6 and n < 100000:
    w = w - 1.0 * logistic(w)[1]; n += 1
print(f"gradient descent (lr 1.0): gradient norm below 1e-6 after {n} iterations")

**Q: Why is this bad for deep learning?** **The Hessian has $O(N^2)$ entries, and inverting it takes $O(N^3)$**, with **N = tens or hundreds of millions** of parameters:

In [ ]:
for n_params in [650, 1e6, 1e8]:
    print(f"N = {n_params:>11,.0f}: Hessian {n_params ** 2 * 4 / 1e9:>14,.3f} GB (float32);  inverting ~{n_params ** 3:.1e} operations")

**Quasi-Newton methods** (BFGS is the most popular) **approximate the inverse Hessian** with cheap low-rank updates from successive gradients ($O(n^2)$ each instead of $O(n^3)$). **L-BFGS** (limited-memory BFGS) **doesn't store the inverse Hessian at all**, only the last few gradient differences. It **usually works very well in full-batch, deterministic mode**, but **doesn't transfer well to the minibatch setting**; making second-order methods work at scale with noisy gradients is active research (K-FAC; Shampoo; 🔎 the Muon optimizer, 2024, a recent popular example). L-BFGS on our full-batch softmax problem, counting loss+gradient evaluations:

In [ ]:
evaluations = {"n": 0}
def loss_and_grad_flat(w_flat):
    # Wraps softmax_loss for scipy: flat vector in, (loss, flat gradient) out.
    evaluations["n"] += 1
    loss, g = softmax_loss(w_flat.reshape(D, C), X_tr, y_tr)
    return loss, g.ravel()

result = minimize(loss_and_grad_flat, np.zeros(D * C), jac=True, method="L-BFGS-B", options={"maxiter": 500, "gtol": 1e-8})
print(f"L-BFGS (full batch): loss {result.fun:.6f} after {evaluations['n']} loss+gradient evaluations")
W_, n = np.zeros((D, C)), 0
while softmax_loss(W_, X_tr, y_tr)[0] > result.fun + 1e-4 and n < 20000:
    W_ -= 2.0 * softmax_loss(W_, X_tr, y_tr)[1]; n += 1
print(f"gradient descent (lr 2.0): {n} evaluations to get within 1e-4 of that loss")

---
## 13. 🔎 Convexity: when the starting point doesn't matter

A function is **convex** if the straight line between any two points on its graph lies above the graph: $f(\theta x + (1-\theta)y)\leq\theta f(x) + (1-\theta)f(y)$. **Every local minimum of a convex function is a global minimum**, so gradient descent can't get trapped. The softmax and SVM losses of a **linear** classifier are convex in $W$ (strictly convex with L2, which makes the minimizer unique). **Neural networks are not convex** (Lecture 5), yet SGD finds good solutions anyway.

**Test:** start the regularized softmax classifier from 5 very different random points:

In [ ]:
finals = []
for seed in range(5):
    W_ = 3.0 * np.random.default_rng(seed).normal(size=(D, C))     # very different starting points
    for _ in range(3000):
        W_ -= 2.0 * softmax_loss(W_, X_tr, y_tr, reg=1e-2)[1]
    finals.append((softmax_loss(W_, X_tr, y_tr, reg=1e-2)[0], W_))
print("final losses from 5 random starts:", np.round([f for f, _ in finals], 5))
print(f"largest distance between the 5 solutions: {max(np.linalg.norm(a - b) for _, a in finals for _, b in finals):.2e}  (the same minimizer)")

And a non-convex 1-D function, $f(x) = x^4 - 3x^2 + x$ (two valleys), from four starting points:

In [ ]:
df = lambda x: 4 * x ** 3 - 6 * x + 1                         # derivative of x^4 - 3x^2 + x
ends = []
for x0 in [-2.0, -0.5, 0.5, 2.0]:
    x_ = x0
    for _ in range(2000):
        x_ -= 0.01 * df(x_)
    ends.append(round(x_, 3))
print("gradient descent from -2, -0.5, 0.5, 2 ends at:", ends, " (two different minima, depending on the start)")

---
## 14. In practice, and in PyTorch

**In practice** (both courses):
* **Adam(W) is a good default choice in many cases**; it often works OK even with a constant learning rate.
* **SGD + Momentum can outperform Adam**, but may need more tuning of the learning rate and schedule.
* **If you can afford full-batch updates, look beyond first-order methods**: try L-BFGS, and **disable all sources of noise**.

In PyTorch, every optimizer has the same interface; a schedule wraps the optimizer:

In [ ]:
# [PyTorch] Needs: pip install torch.
import torch, torch.nn as nn
torch.manual_seed(0)
model = nn.Linear(64, 10)
optimizers = {
    "SGD":          torch.optim.SGD(model.parameters(), lr=1e-2),
    "SGD+Momentum": torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9),
    "Nesterov":     torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9, nesterov=True),
    "AdaGrad":      torch.optim.Adagrad(model.parameters(), lr=1e-2),
    "RMSProp":      torch.optim.RMSprop(model.parameters(), lr=1e-3, alpha=0.99),
    "Adam":         torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999)),
    "AdamW":        torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2),
}
opt = optimizers["AdamW"]
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)    # also StepLR, LinearLR, ...
Xt, yt = torch.tensor(X_tr[:, :64], dtype=torch.float32), torch.tensor(y_tr)
for epoch in range(100):
    loss = nn.functional.cross_entropy(model(Xt), yt)
    opt.zero_grad()          # 1. clear old gradients
    loss.backward()          # 2. compute new gradients (backprop, Lecture 6)
    opt.step()               # 3. update the weights
    scheduler.step()         # 4. update the learning rate
print(f"final loss {loss.item():.4f}")
from torch.autograd import gradcheck                                       # gradient checking, in float64
inp = torch.randn(4, 6, dtype=torch.float64, requires_grad=True)
print("gradcheck(log_softmax):", gradcheck(lambda z: torch.log_softmax(z, dim=1), (inp,)))

---
## 15. Summary

| Topic | Takeaway |
|---|---|
| Random search | Works a little (15.5% on CIFAR-10); hopeless in high dimensions |
| Gradient | Vector of partial derivatives; slope along $u$ is $u^\top\nabla L$; steepest descent is $-\nabla L$ |
| Numeric vs. analytic | Numeric: slow, approximate, easy. Analytic: exact, fast, error-prone. Gradient-check with centered differences in float64 |
| Gradient descent | $w\leftarrow w - \alpha\nabla L$; stable only for $\alpha < 2/\lambda_{\max}$ |
| SGD | Minibatch gradients are unbiased; noise variance ∝ 1/batch size |
| Problems | Poor conditioning (steps ∝ κ), saddle points (common in high dimensions), noise |
| Momentum / Nesterov | Velocity = running mean of gradients; ~√κ steps instead of κ; Nesterov uses the look-ahead gradient |
| AdaGrad / RMSProp | Per-parameter steps from squared-gradient history (summed / decaying) |
| Adam | Momentum + RMSProp + bias correction; β1 0.9, β2 0.999, lr 1e-3 is a strong default |
| AdamW | Decoupled weight decay: L2 ≠ weight decay for Adam |
| Schedules | Step, cosine, linear, inverse sqrt; warmup; scale lr with batch size |
| Second order | Newton: one step on quadratics but O(N²) memory, O(N³) time; L-BFGS for full-batch problems |
| 🔎 Convexity | Linear classifiers: one global minimum; networks: not convex |

### Check yourself
<details><summary>1. Why is the centered difference more accurate than the forward difference?</summary>

In the Taylor expansions of f(x+h) and f(x−h), the h² terms cancel when you subtract, so the error is proportional to h² instead of h.
</details>

<details><summary>2. For gradient descent on a quadratic, what limits the learning rate, and why does that make badly conditioned problems slow?</summary>

Stability needs α < 2/λ_max (the steepest curvature). With such an α, the flattest direction shrinks by only about 1 − 2/κ per step, so the number of steps grows in proportion to κ.
</details>

<details><summary>3. What does Adam's bias correction fix?</summary>

The moment estimates start at zero and are biased toward zero early on (the second moment, with β2 = 0.999, much more so), which makes the first steps too large. Dividing by 1 − β^t removes the bias.
</details>

<details><summary>4. Why are L2 regularization and weight decay the same for SGD but not for Adam?</summary>

With SGD both subtract a multiple of w from the weights. In Adam, the L2 penalty's gradient passes through the adaptive normalization (divided by √v), so its effect depends on each parameter's gradient history; AdamW applies the decay outside the normalization.
</details>

<details><summary>5. Why don't SGD and momentum care whether the problem is rotated, while AdaGrad does?</summary>

Their updates are built only from the gradient, which rotates along with the problem. AdaGrad divides each coordinate by its own history, which depends on how the axes are oriented.
</details>

<details><summary>6. Why isn't Newton's method used for large networks?</summary>

The Hessian has N² entries and inverting it costs O(N³); for N ~ 10⁸ that's impossible. L-BFGS avoids storing it, but works poorly with noisy minibatch gradients.
</details>

### Exercises
1. Add Nesterov in its "change of variables" form to the comparison of §9.
2. Plot GD's and momentum's step counts (§7) against κ on a log-log plot and fit the slopes. Do you get about 1 and 1/2?
3. In §6.2, add Gaussian noise to the gradient and measure the escape time from exactly on the ridge ($y_0 = 0$).
4. In §8.5, find the rotation angle at which AdaGrad's advantage over SGD disappears.
5. Implement linear warmup + cosine decay and train the digits classifier with lr0 = 5 and batch 8. Does warmup help?
6. Implement **AMSGrad** (Adam with $v_t = \max(v_{t-1}, v_t)$) and compare it with Adam on §9's badly conditioned problem.

### References
* CS231n course notes, *Optimization* and *Neural Networks Part 3* (cs231n.github.io/optimization-1, /neural-networks-3).
* Polyak 1964; Nesterov 1983; Sutskever, Martens, Dahl & Hinton 2013.
* Duchi, Hazan & Singer 2011 (AdaGrad); Tieleman & Hinton 2012 (RMSProp); Kingma & Ba 2015 (Adam); Reddi, Kale & Kumar 2018 (AMSGrad); Loshchilov & Hutter 2019 (AdamW).
* Dauphin et al. 2014 (saddle points); Goyal et al. 2017 (large batches, warmup); Loshchilov & Hutter 2017 (cosine).
* Nocedal & Wright, *Numerical Optimization*; Martens & Grosse 2015 (K-FAC); Boyd & Vandenberghe, *Convex Optimization*.

**Next:** Lecture 5, Neural Networks.